# Chạy FKG-E: BRSET 16.266 ảnh / 8.524 bệnh nhân
Chọn **Runtime → Change runtime type → GPU**, rồi chạy lần lượt các ô. Lưới đầy đủ: 108 cấu hình × 5 outer fold; thời gian đầy đủ chưa được đo. Kết quả cũ không đi kèm gói này.


In [ ]:
from pathlib import Path
import sys, subprocess, urllib.request, zipfile, json
url = "https://raw.githubusercontent.com/thanhst/Fuzzy-Knowledge-Graph/main/deliverables/FKGE_BRSET_Tan.zip"
archive = Path("/content/FKGE_BRSET_Tan.zip")
urllib.request.urlretrieve(url, archive)
with zipfile.ZipFile(archive) as z:
    assert all((Path("/content") / name).resolve().is_relative_to(Path("/content")) for name in z.namelist())
    z.extractall("/content")
bundle = Path("/content/FKGE_BRSET_Tan")
release = json.loads((bundle / "RELEASE.json").read_text())
print({key: release[key] for key in ["code_commit", "images", "patients", "outer_folds"]})
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(bundle / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-B", "-X", "utf8", str(bundle / "run.py"), "--check"], check=True)


## Lưu checkpoint trên Google Drive
Cache teacher lớn được giữ trong ổ tạm Colab; mất cache thì tự tính lại. Checkpoint các cấu hình đã hoàn tất nằm trên Drive để tiếp tục sau khi ngắt phiên.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/FKGE_BRSET_Tan_16266")
print("Output:", output)


## Chạy đủ 5 fold
Lệnh tự dùng GPU khi khả dụng. `--resume` tiếp tục cấu hình đã hoàn tất với cùng code/dữ liệu/môi trường/cấu hình. Bảng và ảnh được cập nhật sau mỗi fold trong `reports/`.


In [ ]:
command = [sys.executable, "-B", "-X", "utf8", "-u", str(bundle / "run.py"),
           "--device", "auto", "--output", str(output), "--resume"]
print("Lệnh:", command)
subprocess.run(command, check=True)


## Xem bảng và ảnh đã xuất
Cần hoàn tất đủ 5 outer fold để báo cáo thí nghiệm đầy đủ. AP và PR-AUC hình thang có cột riêng. Các phần teacher và block tính toán không được tính như các mẫu độc lập.


In [ ]:
from IPython.display import Image, display
for name in ["fold_population.png", "metrics_by_fold.png", "metrics_mean_std.png", "roc_pr.png"]:
    path = output / "reports" / name
    if path.exists():
        display(Image(filename=str(path)))
print("Gửi lại thư mục reports, run_manifest.json và environment-freeze.txt trong", output)
